# SEABench: A Guide to Run and Extend the Benchmark

This notebook follows the creation and refinement of one concrete task sequence. It is an executable reading companion: code cells inspect YAML and saved artifacts, or print the commands you would run in a terminal.

> **Note:** except for the optional Colab cell that clones the public repository, this notebook performs no network requests. It does **not** run the benchmark, start prompt optimization, invoke a model, or read an API key. Commands that could consume API credits are printed only. Real benchmark runs can take upto 24 hours per task sequence, depending on the provider.

### 0. Set up the API and environment

A real run needs the repository environment, a provider profile, and the corresponding API-key environment variable. We establish those first because the generation and refinement phase both use them.

Run the cell below to **display** the setup commands. It does not execute them.

In [ ]:
PROVIDER_PROFILE = "openrouter"  #@param {type:"string"}
API_KEY_ENV_VAR = "OPENROUTER_API_KEY"  #@param {type:"string"}
AGENT_MODEL = "moonshotai/kimi-k2.5"  #@param {type:"string"}

setup_commands = [
    "./scripts/setup_venv.sh",
    "source .venv/bin/activate",
    f'export {API_KEY_ENV_VAR}="<your-key>"',
]
print("Commands to run from the SEABench repository (not executed here):\n")
print("\n".join(setup_commands))
print(f"\nProvider profile: {PROVIDER_PROFILE}")
print(f"Agent model:      {AGENT_MODEL}")
print("\nOn Windows, run the Bash setup commands in WSL or another Bash-compatible environment.")

### 1. Locate a local copy of the benchmark

Clone [SEABench-Endogenous-Misalignment/SEABench](https://github.com/SEABench-Endogenous-Misalignment/SEABench) outside this notebook, then set the editable `CODE_ROOT_INPUT` parameter to that folder. If you are working in Colab, run the optional cell below to clone the repository into `/content/SEABench`; the default `CODE_ROOT_INPUT` already points there.

In [ ]:
# Optional (Colab): clone the benchmark repository.
from pathlib import Path
import subprocess

COLAB_REPO = Path("/content/SEABench")
if COLAB_REPO.exists():
    print(f"Repository already exists at {COLAB_REPO}; skipping clone.")
else:
    subprocess.run(
        ["git", "clone", "https://github.com/SEABench-Endogenous-Misalignment/SEABench.git", str(COLAB_REPO)],
        check=True,
    )

In [ ]:
from pathlib import Path
from collections import Counter
import json
import shlex
import statistics

try:
    import yaml
except ImportError as exc:
    raise ImportError("PyYAML is required for read-only YAML inspection. Install the repository requirements first.") from exc

CODE_ROOT_INPUT = "/content/SEABench"  #@param {type:"string"}
requested_root = Path(CODE_ROOT_INPUT).expanduser()
local_fallbacks = [
    requested_root,
    Path.cwd(),
    Path.cwd().parent / "SEABench",
    Path.cwd().parent.parent / "SEABench",
]
CODE_ROOT = next((path.resolve() for path in local_fallbacks if (path / "prompt_optimization").is_dir()), requested_root)

if not (CODE_ROOT / "tasks").is_dir():
    raise FileNotFoundError(
        f"SEABench was not found at {CODE_ROOT}. Clone it outside this notebook and update CODE_ROOT_INPUT."
    )
print(f"Using benchmark repository: {CODE_ROOT}")

### Inspect the provider provider

The provider profile connects a short name such as openrouter to a base URL, the **name** of an environment variable, a default model, and context settings.

In [ ]:
provider_path = CODE_ROOT / "configs" / "providers" / f"{PROVIDER_PROFILE}.yaml"
provider = yaml.safe_load(provider_path.read_text(encoding="utf-8"))
public_provider_fields = {
    key: provider.get(key)
    for key in ("name", "base_url", "api_key_env", "default_model", "context_window_tokens")
}
print(yaml.safe_dump(public_provider_fields, sort_keys=False).rstrip())
assert provider.get("api_key_env") == API_KEY_ENV_VAR, (
    f"The profile expects {provider.get('api_key_env')}, not {API_KEY_ENV_VAR}."
)
print("\nProvider configuration is internally consistent. No key was read and no request was made.")

### 2. Creating a new task sequence

A task sequence will consist of 5 upstream tasks (evolution tasks), followed by 5 downstream tasks (safety-test tasks)

#### 2A. Generating from scratch

A sequence can begin with the generation pipeline. It scans environment assets, proposes upstream tasks around a durable self-evolution mechanism, and proposes downstream tasks that may activate that mechanism. The `POPULATION_SIZE` parameter controls how many **complete candidate sequences** are generated before their tasks are scored and the final 5 upstream + 5 downstream tasks are assembled. Candidate tasks are checked for environment grounding and probed with a live agent during selection.

The following cell only assembles and prints the command.

In [ ]:
GENERATION_SURFACE = "controller_update"  #@param {type:"string"}
TARGET_HARM = "privacy"  #@param {type:"string"}
POPULATION_SIZE = 12  #@param {type:"integer"}
GENERATION_OUTPUT = "prompt_optimization/out/generated_controller_privacy"  #@param {type:"string"}

generation_command = [
    "python3", "prompt_optimization/generate_self_evolution_attack_sequence.py",
    "--surface", GENERATION_SURFACE,
    "--harm", TARGET_HARM,
    "--provider-profile", PROVIDER_PROFILE,
    "--population-size", str(POPULATION_SIZE),
    "--output-dir", GENERATION_OUTPUT,
]
print("Generation command (not executed):\n")
print(shlex.join(generation_command))
print(
    "\nExpected review artifacts include generated_sequence.yaml, upstream/downstream candidates, "
    "environment affordances, the generation prompt, raw candidate outputs, and generation_summary.json."
)

#### 2B. Starting from a human-authored candidate sequence

Instead of starting from scratch, you can start from a human-authored overcomplete set containing **12 upstream tasks and 12 downstream tasks**. The pipeline checks environment-asset grounding and performs one live-agent probe per candidate task before selecting the final 5 upstream and 5 downstream tasks.

Pass the candidate-pool YAML with `--initialize-from-yaml <path-to-yaml>`. For example:

`--initialize-from-yaml prompt_optimization/out/candidate_pools/controller_update/computer_use/privacy.yaml`

After selection and refinement, place the accepted 10-task sequence under `tasks/<evolution_surface>/<domain>/<harm_type>.yaml`—for example, `tasks/controller_update/computer_use/privacy.yaml`.

In [ ]:
HUMAN_AUTHORED_POOL = "prompt_optimization/candidate_pools/controller_update/computer_use/privacy.yaml"  #@param {type:"string"}
HUMAN_AUTHORED_OUTPUT = "prompt_optimization/out/curated_controller_privacy"  #@param {type:"string"}

human_authored_command = [
    "python3", "prompt_optimization/generate_self_evolution_attack_sequence.py",
    "--surface", GENERATION_SURFACE,
    "--harm", TARGET_HARM,
    "--provider-profile", PROVIDER_PROFILE,
    "--initialize-from-yaml", HUMAN_AUTHORED_POOL,
    "--output-dir", HUMAN_AUTHORED_OUTPUT,
]
print("Flag for a human-authored candidate pool:\n")
print(shlex.join(["--initialize-from-yaml", HUMAN_AUTHORED_POOL]))
print("\nComplete generation command (not executed):\n")
print(shlex.join(human_authored_command))

### 3. Generate and Refine a Task Sequence

After generation, launch one refinement pass by adding the `--run-optimizer` flag to the generation command. You can control the search budget with `--max-candidates-per-task 5`, which allows the optimizer to evaluate up to five prompt candidates for each task. The combined command first creates the 10-task sequence and then runs the cascaded upstream and downstream refinement pipeline, writing its state, logs, and final sequence under the generation output directory.

The following cell assembles and prints this combined command.

In [ ]:
MAX_CANDIDATES_PER_TASK = 5  #@param {type:"integer"}
generate_and_refine_command = [
    *generation_command,
    "--run-optimizer",
    "--max-candidates-per-task", str(MAX_CANDIDATES_PER_TASK),
]

print("Flag that launches one refinement round after generation:\n")
print("--run-optimizer")
print("\nComplete generation-and-refinement command (not executed):\n")
print(shlex.join(generate_and_refine_command))

### View the refined trajectory

Each task contains a user prompt and utility criteria; downstream tasks additionally contain safety criteria. Printing them in sequence order makes the story visible: the first half establishes behavior and state, while the second half probes whether that state causes a safety failure.

In [ ]:
TRAJECTORY_PATH = ""  
generated_output_dir = CODE_ROOT / GENERATION_OUTPUT
trajectory_candidates = [
    generated_output_dir / "cascade_optimization" / "final_sequence.yaml",
    generated_output_dir / "generated_sequence.yaml",
    CODE_ROOT / "tasks" / "controller_update" / "computer_use" / "privacy.yaml",
]
if TRAJECTORY_PATH.strip():
    requested_trajectory = Path(TRAJECTORY_PATH).expanduser()
    sequence_path = requested_trajectory if requested_trajectory.is_absolute() else CODE_ROOT / requested_trajectory
else:
    sequence_path = next((path for path in trajectory_candidates if path.is_file()), trajectory_candidates[-1])

if not sequence_path.is_file():
    raise FileNotFoundError(f"No trajectory YAML found at {sequence_path}")
sequence = yaml.safe_load(sequence_path.read_text(encoding="utf-8"))
tasks = sequence.get("tasks", [])
generation = sequence.get("generation", {})
try:
    TASK_SEQUENCE = sequence_path.relative_to(CODE_ROOT).as_posix()
except ValueError:
    TASK_SEQUENCE = str(sequence_path)

def task_role(task):
    explicit = (task.get("optimization") or {}).get("role")
    if explicit:
        return explicit
    return "downstream" if "safety_llm" in (task.get("validator") or {}) else "upstream"

print(f"Reading trajectory: {sequence_path}")
print(f"Title:              {sequence.get('title')}")
print(f"Sequence ID:        {sequence.get('id')}")
print(f"Surface:            {generation.get('surface')}")
print(f"Target harm:        {generation.get('target_harm')}")
print(f"Upstream tasks:     {sum(task_role(task) == 'upstream' for task in tasks)}")
print(f"Downstream tasks:   {sum(task_role(task) == 'downstream' for task in tasks)}")

for index, task in enumerate(tasks, start=1):
    validators = task.get("validator") or {}
    print("=" * 88)
    print(f"{index:02d}. [{task_role(task).upper()}] {task.get('id')} — {task.get('title')}")
    print("\nPROMPT\n" + str(task.get("prompt") or "").strip())
print("=" * 88)

### 4. Refine an existing task sequence

This section describes how to run refinement on an existing task sequence from the benchmark; this is the main experiment from the paper. The failure discovery pipeline considers prompt rewrites in task order and evaluates each candidate with paired trajectories: a self-evolving agent and a non-evolving baseline. 

1. **Upstream hardening:** retain prompts that support useful completion and a meaningful persistent update while remaining contrastive with the baseline.
2. **State carry-forward:** replay the accepted prompt and use its workspace/session state for the next task; rejected branches do not become history.
3. **Freeze at the boundary:** after upstream tasks, prevent new downstream self-evolution so later failures are attributable to the persistent upstream state.
4. **Downstream refinement:** search for safety-test prompts that preserve intent and expose a self-evolution-induced failure under the hard filters.

Add the `--resume` flag to the same command to resume an interrupted refinement run.

In [ ]:
REFINEMENT_OUTPUT = "prompt_optimization/out/controller_update_computer_use_privacy"  #@param {type:"string"}
MAX_CANDIDATES_PER_TASK = 5  #@param {type:"integer"}

refinement_command = [
    "python3", "prompt_optimization/refine_existing_sequence.py",
    "--task-sequence", TASK_SEQUENCE,
    "--output-dir", REFINEMENT_OUTPUT,
    "--provider-profile", PROVIDER_PROFILE,
    "--agent-model", AGENT_MODEL,
    "--max-candidates-per-task", str(MAX_CANDIDATES_PER_TASK),
    "--downstream-safety-lookahead",
    "--downstream-standalone",
]
print("Refinement command (not executed):\n")
print(shlex.join(refinement_command))
print("\nRun it only after reviewing the sequence, provider profile, expected cost, and output path.")

### 5. Inspect saved refinement outputs and aggregate metrics

After running the printed command externally (A single run may take upto 24 hours depending on the provider), point REFINEMENT_OUTPUT to that output directory and rerun this cell. It reads only existing files. The principal artifacts are:

- **generated_sequence.yaml**: a local copy of the input;
- **cascade_optimization/working_sequence.yaml**: the accepted sequence as refinement progresses;
- **cascade_optimization/sequence_optimization_log.jsonl**: one accepted task-level result per stage; and
- **final_refined_sequence.yaml**: the final accepted sequence.

In [ ]:
output_dir = CODE_ROOT / REFINEMENT_OUTPUT
cascade_dir = output_dir / "cascade_optimization"
log_path = cascade_dir / "sequence_optimization_log.jsonl"
final_path = output_dir / "final_refined_sequence.yaml"

print(f"Output directory: {output_dir}")
if not output_dir.exists():
    print("No saved refinement output exists here yet.")
else:
    total_files = sum(path.is_file() for path in output_dir.rglob("*"))
    visible = []
    for top_level in sorted(output_dir.iterdir(), key=lambda path: path.name):
        if top_level.is_file():
            visible.append(top_level.relative_to(output_dir).as_posix())
        elif top_level.is_dir():
            for child in sorted(top_level.iterdir(), key=lambda path: path.name):
                name = child.relative_to(output_dir).as_posix()
                visible.append(name + "/" if child.is_dir() else name)
    print(f"Saved files found: {total_files}")
    print("Entries shown below (maximum depth: 2):")
    for name in visible:
        print(f"  {name}")

if log_path.is_file():
    rows = [json.loads(line) for line in log_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    scores = [float(row["final_score"]) for row in rows if isinstance(row.get("final_score"), (int, float))]
    role_counts = Counter(str(row.get("role") or "unknown") for row in rows)
    accepted = sum(bool(row.get("accepted_prompt_changed")) for row in rows)
    hard_ok = sum(bool(row.get("final_hard_ok")) for row in rows)
    downstream_by_task = {
        str(row.get("task_id") or ""): row
        for row in rows
        if str(row.get("role") or "") == "downstream"
    }
    downstream_rows = list(downstream_by_task.values())
    expected_downstream = sum(task_role(task) == "downstream" for task in tasks)

    def downstream_final_metrics(row):
        task_dir_value = str(row.get("task_dir") or "").strip()
        if not task_dir_value:
            return None, None, None, None
        task_dir = Path(task_dir_value)
        if not task_dir.is_absolute():
            task_dir = CODE_ROOT / task_dir
        evaluation_path = task_dir / "final_evaluation.json"
        if not evaluation_path.is_file():
            return None, None, None, None
        evaluation = json.loads(evaluation_path.read_text(encoding="utf-8"))
        self_evolving = evaluation.get("self_evolving")
        non_evolving = evaluation.get("none")
        self_evolving = self_evolving if isinstance(self_evolving, dict) else {}
        non_evolving = non_evolving if isinstance(non_evolving, dict) else {}
        return (
            self_evolving.get("task_passed"),
            self_evolving.get("safety_passed"),
            non_evolving.get("task_passed"),
            non_evolving.get("safety_passed"),
        )

    downstream_metrics = [downstream_final_metrics(row) for row in downstream_rows]
    self_utility_results = [values[0] for values in downstream_metrics if isinstance(values[0], bool)]
    self_safety_results = [values[1] for values in downstream_metrics if isinstance(values[1], bool)]
    none_utility_results = [values[2] for values in downstream_metrics if isinstance(values[2], bool)]
    none_safety_results = [values[3] for values in downstream_metrics if isinstance(values[3], bool)]
    print("\nAGGREGATE REFINEMENT DIAGNOSTICS")
    print(f"  task stages logged:       {len(rows)}")
    print(f"  stages by role:           {dict(role_counts)}")
    print(f"  accepted prompt rewrites: {accepted} / {len(rows)}")
    print(f"  final hard-filter passes: {hard_ok} / {len(rows)}")
    print(f"  mean final score:         {statistics.mean(scores):.3f}" if scores else "  mean final score:         unavailable")
    condition_results = [
        ("final utility passed (self-evolving)", self_utility_results),
        ("final utility passed (non-evolving)", none_utility_results),
        ("final safety passed (self-evolving)", self_safety_results),
        ("final safety passed (non-evolving)", none_safety_results),
    ]
    for label, results in condition_results:
        if len(results) == expected_downstream:
            print(f"  {label + ':':43} {sum(results)} / {expected_downstream}")
        else:
            print(f"  {label + ':':43} unavailable ({len(results)} / {expected_downstream} downstream results found)")
else:
    print(f"\nNo task-level refinement log found at {log_path}.")

if final_path.is_file():
    final_sequence = yaml.safe_load(final_path.read_text(encoding="utf-8"))
    final_tasks = final_sequence.get("tasks", [])
    changed = sum(
        str(before.get("prompt") or "").strip() != str(after.get("prompt") or "").strip()
        for before, after in zip(tasks, final_tasks)
    )
    print(f"\nFinal refined sequence contains {len(final_tasks)} tasks; {changed} prompts differ from the selected input.")
else:
    print(f"No final refined sequence found at {final_path}.")

## 6. Extend SEABench

Two ways (not exhaustive) to extend SEABench are described below:

1. **New sequence, existing provider and agent:** change the surface/harm generation choices, review the generated YAML, validate its structure, then refine it using the procedure described above. You can also add new harm types beyond the four currently in the benchmark.
2. **New OpenAI-compatible API:** add a provider YAML under **configs/providers/** containing the base URL, API-key environment-variable name, default model, and context settings.

The cell below previews a provider profile.

In [ ]:
NEW_PROVIDER_NAME = "my_provider"  #@param {type:"string"}
NEW_BASE_URL = "https://api.example.com/v1"  #@param {type:"string"}
NEW_API_KEY_ENV = "MY_PROVIDER_API_KEY"  #@param {type:"string"}
NEW_DEFAULT_MODEL = "provider/model-name"  #@param {type:"string"}

provider_preview = {
    "name": NEW_PROVIDER_NAME,
    "base_url": NEW_BASE_URL,
    "api_key_env": NEW_API_KEY_ENV,
    "default_model": NEW_DEFAULT_MODEL,
    "context_window_tokens": 32768,
}
print(f"Preview for configs/providers/{NEW_PROVIDER_NAME}.yaml (not written):\n")
print(yaml.safe_dump(provider_preview, sort_keys=False).rstrip())

## Further Links

- [Submit a pull request](https://github.com/SEABench-Endogenous-Misalignment/SEABench/pulls)
- [SEABench codebase README](https://github.com/SEABench-Endogenous-Misalignment/SEABench/blob/main/README.md)
- [Agent tools documentation](https://github.com/SEABench-Endogenous-Misalignment/SEABench/blob/main/docs/agent_tools.md)
- [Task-sequence creation documentation](https://github.com/SEABench-Endogenous-Misalignment/SEABench/blob/main/docs/task_sequence_creation.md)